# 06 – Integrated Risk Engine

Combine molecular toxicity predictions and FAERS evidence into a transparent risk-prioritization framework. The engine separates evidence streams and records assumptions so a high score can be traced to its drivers.

**Critical limitation:** the integrated score is a ranking/triage construct, not a probability of clinical harm or a causal estimate.

In [1]:
from pathlib import Path
import sys, json, joblib
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parent
if not (ROOT/"data").exists(): ROOT = Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

from src.risk_engine import toxicity_component, build_integrated_score, classify_risk

TOX = ROOT/"data/processed/tox21"
FAERS = ROOT/"data/processed/faers"
X = pd.read_parquet(TOX/"X_tox21.parquet")
signals = pd.read_parquet(FAERS/"faers_drug_ae_signals.parquet")

In [2]:
KEY_ENDPOINTS = ["NR-AR","NR-AhR","NR-ER","NR-PPAR-gamma","SR-ARE","SR-p53"]

def model_file(endpoint):
    return ROOT/"models/tox21"/f"{endpoint.replace('/','_').replace('-','_')}.joblib"

candidate_X = X.sample(min(5,len(X)), random_state=42)
pred = {}
for endpoint in KEY_ENDPOINTS:
    p = model_file(endpoint)
    if p.exists():
        model = joblib.load(p)
        pred[endpoint] = model.predict_proba(candidate_X)[:,1]

candidate_pred = pd.DataFrame(pred, index=candidate_X.index)
candidate_pred["toxicity_component"] = toxicity_component(candidate_pred)
candidate_pred["faers_component"] = 0.0
candidate_pred["seriousness_component"] = 0.0
candidate_pred["risk_score"] = build_integrated_score(
    candidate_pred["toxicity_component"],
    candidate_pred["faers_component"],
    candidate_pred["seriousness_component"]
).values
candidate_pred["risk_class"] = candidate_pred["risk_score"].map(classify_risk)
display(candidate_pred.sort_values("risk_score", ascending=False))

,NR-AR,NR-AhR,NR-ER,NR-PPAR-gamma,SR-ARE,SR-p53,toxicity_component,faers_component,seriousness_component,risk_score,risk_class
4567,0.040144,0.156109,0.262337,0.269173,0.354442,0.238390,0.220099,0.0,0.0,0.121055,Lower
1677,0.108791,0.104381,0.293693,0.141043,0.285755,0.215269,0.191488,0.0,0.0,0.105319,Lower
1142,0.099648,0.110487,0.201283,0.077916,0.094879,0.042420,0.104439,0.0,0.0,0.057441,Lower
4958,0.085863,0.054993,0.135813,0.013139,0.121986,0.088502,0.083383,0.0,0.0,0.045860,Lower
2592,0.065455,0.044331,0.122374,0.029255,0.097927,0.053485,0.068804,0.0,0.0,0.037842,Lower


## Known-drug FAERS evidence

A known drug can receive a FAERS-led evidence score. Mapping FAERS evidence to a new molecule requires defensible drug identity/class/target mapping; it should never be silently assumed.

In [3]:
drug = signals.groupby("suspect_drug").agg(
    max_prr=("PRR","max"),
    max_chi2=("chi2","max"),
    signal_count=("evans_signal","sum")
).reset_index()

drug["faers_strength"] = (
    np.log1p(drug["max_prr"].replace(np.inf,np.nan).fillna(0)) +
    0.25*np.log1p(drug["max_chi2"].fillna(0)) +
    0.5*np.log1p(drug["signal_count"])
)

def minmax(s):
    return (s-s.min())/(s.max()-s.min()) if s.max()!=s.min() else s*0

drug["faers_component"] = minmax(drug["faers_strength"])
drug["seriousness_component"] = 0.0
drug["risk_score"] = build_integrated_score(
    0.0, drug["faers_component"], drug["seriousness_component"]
).values
drug["risk_class"] = drug["risk_score"].map(classify_risk)
display(drug.sort_values("risk_score", ascending=False).head(30))

,suspect_drug,max_prr,max_chi2,signal_count,faers_strength,faers_component,seriousness_component,risk_score,risk_class
0,%20 MANNITOL SOL?SYON CAM,87999.833333,75427.714284,0,14.192837,0.848151,0.0,0.254445,Lower
1,(DAUNORUBICIN AND CYTARABINE) LIPOSOME,3692.230769,3426.733969,3,10.942318,0.613661,0.0,NaN,Higher
2,"(TS) ADEFURONIC TABLET 25MG, TAB, 25MG",5443.288660,5386.765304,0,10.750294,0.599808,0.0,NaN,Higher
3,*JNJ-56022473,1051.790837,1048.703775,0,8.698266,0.451776,0.0,NaN,Higher
4,.ALPHA.-PYRROLIDINOVALEROTHIOPHENONE,5415.333333,10299.336696,5,11.803037,0.675752,0.0,NaN,Higher
5,.ALPHA.1-PROTEINASE INHIBITOR HUMAN,3615.438356,1806.726160,54,12.071868,0.695146,0.0,NaN,Higher
6,.DELTA.8-TETRAHYDROCANNABINOL\HERBALS,21119.960000,20306.730767,0,12.437710,0.721537,0.0,NaN,Higher
7,0.9% SODIUM CHLORIDE INJECTION,18856.142857,9427.107164,3,12.825657,0.749523,0.0,NaN,Higher
8,"0.9% SODIUM CHLORIDE INJECTION, USP",3161.670659,3141.863093,0,10.072394,0.550905,0.0,NaN,Higher
9,"0.9% SODIUM CHLORIDE INJECTION, USP IN MINI-BA...",43999.833333,37712.928573,0,13.326410,0.785647,0.0,NaN,Higher


In [4]:
OUT = ROOT/"data/processed/integrated"
OUT.mkdir(parents=True, exist_ok=True)
candidate_pred.to_parquet(OUT/"hypothetical_candidate_risk.parquet")
drug.to_parquet(OUT/"drug_level_faers_risk.parquet")

metadata = {
    "weights":{"toxicity":0.55,"faers":0.30,"seriousness":0.15},
    "interpretation":"Ranking/triage score; not clinical probability or causal estimate.",
    "next_steps":[
        "validated drug-target/class mapping",
        "mechanistic pathway weights",
        "internal assay integration",
        "uncertainty quantification",
        "prospective validation"
    ]
}
with open(OUT/"risk_engine_metadata.json","w") as f: json.dump(metadata,f,indent=2)
print("Integrated artifacts saved.")

Integrated artifacts saved.


## Hiring-manager narrative

ImmunoToxAI demonstrates a complete safety-analytics architecture: molecular representation → pathway toxicity prediction → pharmacovigilance signal detection → interpretable evidence integration.

A real deployment would use these outputs to prioritize confirmatory assays, deepen pharmacovigilance review, compare portfolio candidates, and create an auditable safety evidence profile. Final decisions remain with qualified scientific and clinical experts.